In [2]:
import os
import pandas as pd

In [16]:
dir_input = "../data/input/movie_lens" 
dir_output = "../data/outputs" 

### Step 1: Read Files

In [6]:
df_movies_data = pd.read_csv(os.path.join(dir_input,"movies.csv"))
df_movies_data.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
df_movies_data.columns


Index(['movieId', 'title', 'genres'], dtype='str')

In [7]:
df_ratings = pd.read_csv(os.path.join(dir_input, 'ratings.csv'))
df_ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [8]:
df_tags = pd.read_csv(os.path.join(dir_input, 'tags.csv'))
df_tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


### Step 2: Aggregate Movie Ratings
We need to calculate the average rating and the number of ratings for each movie. We group the ratings table by movieId and apply the mean and count functions.


In [9]:
# Aggregate ratings: Calculate mean and count per movieId
rating_stats = df_ratings.groupby('movieId')['rating'].agg(['mean', 'count']).reset_index()
rating_stats.head()

,movieId,mean,count
0,1,3.920930,215
1,2,3.431818,110
2,3,3.259615,52
3,4,2.357143,7
4,5,3.071429,49


### Step 3: Consolidate User Tags
Since a movie can have multiple tags across different users, we consolidate them into a single string (separated by a pipe |) for each movie.

In [8]:
# Group tags by movieId and join them into a single string
movie_tags = df_tags.groupby('movieId')['tag'].apply(lambda x: '|'.join(x)).reset_index()
movie_tags.rename(columns={'tag': 'tags'}, inplace=True)

print("Consolidated Tags per Movie:")
print(movie_tags.head())

Consolidated Tags per Movie:
   movieId                                          tags
0        1                               pixar|pixar|fun
1        2  fantasy|magic board game|Robin Williams|game
2        3                                     moldy|old
3        5                              pregnancy|remake
4        7                                        remake


### Step 4: Create the Analytics Base Table (ABT)
Now, we merge the movies table with our rating_stats and movie_tags using the movieId as the primary key.

In [9]:
# Merge movies with ratings stats
abt = pd.merge(df_movies_data, rating_stats, on='movieId', how='left')

# Merge the result with the consolidated tags
abt = pd.merge(abt, movie_tags, on='movieId', how='left')
abt.head()

,movieId,title,genres,mean,count,tags
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215.0,pixar|pixar|fun
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.431818,110.0,fantasy|magic board game|Robin Williams|game
2,3,Grumpier Old Men (1995),Comedy|Romance,3.259615,52.0,moldy|old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.357143,7.0,NaN
4,5,Father of the Bride Part II (1995),Comedy,3.071429,49.0,pregnancy|remake


In [18]:
# abt['mean'] = abt['mean'].round(1)

In [19]:
abt = abt.rename(columns={
 "mean": "avg_rating",
 "count": "number_of_viewers"
})

abt.head()

,movieId,title,genres,avg_rating,number_of_viewers
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.431818,110.0
2,3,Grumpier Old Men (1995),Comedy|Romance,3.259615,52.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.357143,7.0
4,5,Father of the Bride Part II (1995),Comedy,3.071429,49.0


In [20]:
df_links = pd.read_csv(os.path.join(dir_input, 'links.csv'), 
 dtype={"movieId": "int",
 "imdbId": "string",
 "tmdbId": "string"})
df_links.head()

,movieId,imdbId,tmdbId
0,1,0114709,862
1,2,0113497,8844
2,3,0113228,15602
3,4,0114885,31357
4,5,0113041,11862


In [21]:
# Merge movies with links
final_abt = pd.merge(abt, df_links, on='movieId', how='left')

final_abt.head()

,movieId,title,genres,avg_rating,number_of_viewers,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215.0,0114709,862
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.431818,110.0,0113497,8844
2,3,Grumpier Old Men (1995),Comedy|Romance,3.259615,52.0,0113228,15602
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.357143,7.0,0114885,31357
4,5,Father of the Bride Part II (1995),Comedy,3.071429,49.0,0113041,11862


In [22]:
final_abt.to_csv(os.path.join(dir_output, 'analytics_base_table.csv'))